# Task 3 audio tagging via fine-tuned PANNs CNN14 (Colab GPU)

Fine-tunes a CNN14 backbone pretrained on AudioSet (mAP 0.431) on the assignment's 10-class tagging set and writes `predictions3.json`.

**Before running:**
1. Upload `student_files_updated.zip` to your Google Drive at `/content/drive/MyDrive/CSE153/student_files_updated.zip` (same place as the previous notebook).
2. Runtime menu, set GPU runtime (A100 or V100 preferred, T4 still works).
3. Run the cells top to bottom.

Saves `predictions3.json` to the Colab working directory and to your Drive at `/content/drive/MyDrive/CSE153/predictions3.json`.

## 1. GPU check and dependency install

In [ ]:
!nvidia-smi

In [ ]:
# panns_inference gives us the pretrained CNN14 backbone class plus the published checkpoint loader.
# The rest (torch, torchaudio, librosa, sklearn, numpy) ships with Colab.
!pip -q install panns_inference
import torch, torchaudio, librosa, sklearn, numpy as np
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Mount Drive and unzip the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil
ZIP_PATH = '/content/drive/MyDrive/CSE153/student_files_updated.zip'
WORKDIR = '/content/work'
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH}; upload it to Drive or change ZIP_PATH'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
if not os.path.isdir('student_files'):
    !unzip -q -o {ZIP_PATH} -d {WORKDIR}
    if os.path.isdir(os.path.join(WORKDIR, '__MACOSX')):
        shutil.rmtree(os.path.join(WORKDIR, '__MACOSX'))
print('Contents:', os.listdir('student_files'))
print('Train clips:', len(os.listdir('student_files/task3_audio_classification/train')))
print('Test clips:', len(os.listdir('student_files/task3_audio_classification/test')))

## 3. Download the pretrained CNN14 checkpoint

In [ ]:
# Published checkpoint from the PANNs paper, AudioSet pretraining at mAP 0.431.
# About 326 MB; cached in /content so a runtime restart reuses it.
import os, urllib.request
CKPT_PATH = '/content/Cnn14_mAP=0.431.pth'
CKPT_URL  = 'https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth?download=1'
if not os.path.exists(CKPT_PATH):
    print('downloading checkpoint ...')
    urllib.request.urlretrieve(CKPT_URL, CKPT_PATH)
print('checkpoint:', CKPT_PATH, os.path.getsize(CKPT_PATH) // (1024*1024), 'MB')

## 4. Dataset, model, mixup, evaluation

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

DATAROOT = 'student_files/task3_audio_classification'
SAMPLE_RATE = 32000           # PANNs native sample rate
AUDIO_DURATION = 10
N_CLASSES = 10
BATCH_SIZE = 32               # fine-tuning the full backbone, smaller batch fits A100 comfortably
EPOCHS = 25
BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
MIXUP_ALPHA = 0.4
TAGS = ['rock', 'oldies', 'jazz', 'pop', 'dance', 'blues', 'punk', 'chill', 'electronic', 'country']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

torch.manual_seed(0); np.random.seed(0); random.seed(0)

In [ ]:
def load_waveform(path):
    # PANNs takes raw mono audio at 32 kHz; the model has its own log-mel front-end internally.
    waveform, _ = librosa.load(os.path.join(DATAROOT, path), sr=SAMPLE_RATE)
    target_len = SAMPLE_RATE * AUDIO_DURATION
    if len(waveform) < target_len:
        waveform = np.pad(waveform, (0, target_len - len(waveform)))
    else:
        waveform = waveform[:target_len]
    return torch.from_numpy(waveform.astype(np.float32))

class TaggingDataset(Dataset):
    def __init__(self, meta, preload=True):
        self.meta = meta
        self.ids = list(meta.keys())
        self.cache = {}
        if preload:
            for path in tqdm(self.ids, desc='loading waveforms'):
                self.cache[path] = load_waveform(path)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        path = self.ids[idx]
        wave = self.cache[path] if path in self.cache else load_waveform(path)
        label = torch.tensor([1 if t in self.meta[path] else 0 for t in TAGS], dtype=torch.float32)
        return wave, label, path

In [ ]:
from panns_inference.models import Cnn14

class Cnn14Tagger(nn.Module):
    """PANNs CNN14 backbone with a fresh 10-class head on the 2048-d embedding."""
    def __init__(self, ckpt_path, n_classes=N_CLASSES):
        super().__init__()
        self.backbone = Cnn14(sample_rate=SAMPLE_RATE, window_size=1024, hop_size=320,
                              mel_bins=64, fmin=50, fmax=14000, classes_num=527)
        state = torch.load(ckpt_path, map_location='cpu')
        self.backbone.load_state_dict(state['model'])
        # The published Cnn14 keeps its 527-class fc_audioset; we ignore it and read the 2048-d embedding instead.
        self.head = nn.Sequential(
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, n_classes),
        )
    def forward(self, x):
        out = self.backbone(x)              # x: (B, T) raw waveform at 32 kHz
        return self.head(out['embedding'])  # logits over our 10 classes

def mixup_batch(x, y, alpha):
    # Simple mixup: blend two random samples in each batch and blend their multi-hot labels accordingly.
    if alpha <= 0:
        return x, y
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], lam * y + (1 - lam) * y[perm]

def evaluate(model, loader):
    model.eval()
    logits_all, target_all, paths_all = [], [], []
    with torch.no_grad():
        for x, y, paths in loader:
            x = x.to(DEVICE, non_blocking=True)
            logits_all.append(model(x).cpu())
            target_all.append(y)
            paths_all += list(paths)
    logits = torch.cat(logits_all)
    probs = torch.sigmoid(logits).numpy()
    targets = torch.cat(target_all).numpy()
    mAP = None
    if targets.sum() > 0:
        try:
            mAP = average_precision_score(targets, probs, average='macro')
        except Exception:
            mAP = None
    return probs, paths_all, mAP

## 5. Build loaders, optimizer, schedule

In [ ]:
train_meta = eval(open(os.path.join(DATAROOT, 'train.json')).read())
test_meta  = {k: [] for k in eval(open(os.path.join(DATAROOT, 'test.json')).read())}
print('Train:', len(train_meta), 'Test:', len(test_meta))

full_train = TaggingDataset(train_meta, preload=True)
g = torch.Generator().manual_seed(0)
n = len(full_train)
n_tr = int(n * 0.9); n_va = n - n_tr
tr_sub, va_sub = random_split(full_train, [n_tr, n_va], generator=g)
te = TaggingDataset(test_meta, preload=True)

loader_tr = DataLoader(tr_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
loader_va = DataLoader(va_sub, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
loader_te = DataLoader(te,     batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
model = Cnn14Tagger(CKPT_PATH).to(DEVICE)
# Different learning rates for the pretrained backbone vs the fresh classifier head:
# the backbone is already a good feature extractor and needs only a small nudge.
opt = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': BACKBONE_LR},
    {'params': model.head.parameters(),     'lr': HEAD_LR},
], weight_decay=1e-4)
crit = nn.BCEWithLogitsLoss()
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

## 6. Train a small seed ensemble

Three seeds (0, 1, 2) trained back to back. Each seed uses the same train/val split (the split generator is independent of the global seed) but a different model init, mixup draw, and DataLoader shuffle order, so their best-checkpoint state dicts are different stochastic gradient outcomes of the same recipe. Averaging their test predictions later is what tends to give a small leaderboard bump on top of any single run.

In [ ]:
N_SEEDS = 3   # seeds 0, 1, 2

def train_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    m = Cnn14Tagger(CKPT_PATH).to(DEVICE)
    op = torch.optim.AdamW([
        {'params': m.backbone.parameters(), 'lr': BACKBONE_LR},
        {'params': m.head.parameters(),     'lr': HEAD_LR},
    ], weight_decay=1e-4)
    sc = torch.optim.lr_scheduler.CosineAnnealingLR(op, T_max=EPOCHS)
    cr = nn.BCEWithLogitsLoss()
    best, best_state = -1.0, None
    for ep in range(EPOCHS):
        m.train()
        running, nb = 0.0, 0
        for x, y, _ in tqdm(loader_tr, desc=f'seed{seed} ep{ep+1}/{EPOCHS}', leave=False):
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            x_mix, y_mix = mixup_batch(x, y, MIXUP_ALPHA)
            op.zero_grad()
            loss = cr(m(x_mix), y_mix)
            loss.backward(); op.step()
            running += loss.item(); nb += 1
        sc.step()
        _, _, vmap = evaluate(m, loader_va)
        print(f'  seed{seed} ep{ep+1} loss={running/nb:.4f} val_mAP={vmap:.4f}')
        if vmap is not None and vmap > best:
            best = vmap
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
    print(f'seed {seed} done. best val_mAP={best:.4f}')
    return best_state, best

states, vmaps = [], []
for s in range(N_SEEDS):
    st, vm = train_seed(s)
    states.append(st)
    vmaps.append(vm)
print('per-seed best val_mAPs:', [f'{v:.4f}' for v in vmaps], 'mean:', f'{sum(vmaps)/len(vmaps):.4f}')

## 7. Test-time augmentation, ensemble, write predictions3.json

For each seed's best checkpoint, average sigmoid predictions over five circular time-shifts of the input waveform (about -1.0, -0.5, 0.0, +0.5, +1.0 seconds). Then average those per-seed prediction matrices into a single ensemble matrix and write `predictions3.json`.

In [ ]:
SHIFTS_SECONDS = (-1.0, -0.5, 0.0, 0.5, 1.0)   # circular time shifts averaged at inference

def predict_with_tta(m, loader, shifts_seconds=SHIFTS_SECONDS):
    m.eval()
    shifts = [int(s * SAMPLE_RATE) for s in shifts_seconds]
    all_probs, all_paths = [], []
    with torch.no_grad():
        for x, _, paths in loader:
            x = x.to(DEVICE, non_blocking=True)
            probs_sum = torch.zeros(x.size(0), N_CLASSES, device=DEVICE)
            for sh in shifts:
                xs = x if sh == 0 else torch.roll(x, shifts=sh, dims=-1)
                probs_sum += torch.sigmoid(m(xs))
            all_probs.append((probs_sum / len(shifts)).cpu())
            all_paths += list(paths)
    return torch.cat(all_probs).numpy(), all_paths

per_seed_probs, paths = [], None
for s, st in enumerate(states):
    model.load_state_dict(st)
    probs_s, p_s = predict_with_tta(model, loader_te)
    per_seed_probs.append(probs_s)
    paths = p_s
    print(f'seed {s} TTA inference done')

probs_ensemble = np.mean(per_seed_probs, axis=0)

preds = {(p[2:] if p.startswith('./') else p):
         {TAGS[j]: float(probs_ensemble[i][j]) for j in range(N_CLASSES)}
         for i, p in enumerate(paths)}

out_local = '/content/work/predictions3.json'
with open(out_local, 'w') as f:
    f.write(repr(preds) + '\n')
print('Wrote', out_local, 'entries=', len(preds))

import shutil
drive_out = '/content/drive/MyDrive/CSE153/predictions3.json'
os.makedirs(os.path.dirname(drive_out), exist_ok=True)
shutil.copy(out_local, drive_out)
print('Saved copy to', drive_out)

In [ ]:
# Quick sanity check on the output format
d = eval(open(out_local).read())
print('len:', len(d))
sample_k = list(d.keys())[0]
print('sample key:', repr(sample_k))
print('sample value:', d[sample_k])
assert all(isinstance(v, dict) and len(v) == N_CLASSES for v in d.values()), 'malformed predictions'
print('OK')

Download `/content/work/predictions3.json` from the Colab file panel, or pull it from your Drive at `/content/drive/MyDrive/CSE153/predictions3.json`. Place it alongside `predictions1.json` and `predictions2.json` in the assignment directory and submit all three plus `assignment1.py` and `writeup.txt`.